In [ ]:
import configparser

import pandas as pd

In [ ]:
from machine_learning.prepareInput import prepareInput

In [ ]:
data = configparser.ConfigParser(interpolation=configparser.ExtendedInterpolation())
data.read('../data.ini')
data = data['DEFAULT']

In [ ]:
samples_RAS = pd.read_csv('../' + data['all_noHypoHyper_RAS___samples'], sep='\t', engine='python')['SAMPLE_ID']
samples_nonRAS = pd.read_csv('../' + data['all_noHypoHyper_nonRAS___samples'], sep='\t', engine='python')['SAMPLE_ID']

# merged dataset

In [ ]:
mutations_merged = pd.read_csv('../' + data['all_noHypoHyper___mutations'], sep='\t', engine='python', comment='#')
mutations_merged = mutations_merged[~mutations_merged['Hugo_Symbol'].isin(['KRAS', 'HRAS', 'NRAS'])]

## only mutational data

### no clusters

In [ ]:
onlyGenes_mut_merged = mutations_merged.rename(columns={'Tumor_Sample_Barcode' : 'Sample', 'Hugo_Symbol' : 'name', 'Variant_Classification': 'var'})
onlyGenes_mut_merged = onlyGenes_mut_merged[['Sample', 'name', 'var']]

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    mutations=onlyGenes_mut_merged,
    trainPercentage=float(data['train_fraction']),
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}merged_mut/train_RAS", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}merged_mut/train_nonRAS", index=False)
test_RAS.to_csv(f"../{data['ml___input']}merged_mut/test_RAS", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}merged_mut/test_nonRAS", index=False)

Save which samples were divided in train and test to use the same splits for the other types of input

In [ ]:
samples_train = train_RAS.index.append(train_nonRAS.index)
samples_test = test_RAS.index.append(test_nonRAS.index)

### with Mutclust's clusters

In [ ]:
if 'mutclust_mut_merged' not in locals():
    mutclust_mut_merged = mutations_merged.rename(columns={'Tumor_Sample_Barcode' : 'Sample', 'Hugo_Symbol' : 'name', 'Transcript_ID': 'merge', 'Protein_position' : 'start', 'Variant_Classification': 'var'})
    mutclust_mut_merged['end'] = mutclust_mut_merged['start']
    mutclust_mut_merged = mutclust_mut_merged[['Sample', 'name', 'merge', 'start', 'end', 'var']]

if 'mutclust_clust' not in locals():
    mutclust_clust = pd.read_csv('../' + data['mutclustsw___all_noHypoHyper___out'], sep='\t', engine='python', comment='#')
    mutclust_clust = mutclust_clust.rename(columns={'Gene' : 'merge', 'Start' : 'start', 'End' : 'end', 'Pvalue' : 'P'})
    mutclust_clust = mutclust_clust[['merge', 'start', 'end', 'P']]

#### without variants - with both clusters and non-clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=mutclust_mut_merged,
    includeVariants=False,
    clusters=mutclust_clust,
    clusters_threshold=float(data['mutclustSignificanceThreshold']),
    onlyClusters=False,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}merged_mut_[\"mutclust - ALL\"]/train_RAS", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}merged_mut_[\"mutclust - ALL\"]/train_nonRAS", index=False)
test_RAS.to_csv(f"../{data['ml___input']}merged_mut_[\"mutclust - ALL\"]/test_RAS", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}merged_mut_[\"mutclust - ALL\"]/test_nonRAS", index=False)

#### without variants - with only clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=mutclust_mut_merged,
    includeVariants=False,
    clusters=mutclust_clust,
    clusters_threshold=float(data['mutclustSignificanceThreshold']),
    onlyClusters=True,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}merged_mut_[\"onlyMutclust - ALL\"]/train_RAS", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}merged_mut_[\"onlyMutclust - ALL\"]/train_nonRAS", index=False)
test_RAS.to_csv(f"../{data['ml___input']}merged_mut_[\"onlyMutclust - ALL\"]/test_RAS", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}merged_mut_[\"onlyMutclust - ALL\"]/test_nonRAS", index=False)

#### with variants - with both clusters and non-clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=mutclust_mut_merged,
    includeVariants=True,
    clusters=mutclust_clust,
    clusters_threshold=float(data['mutclustSignificanceThreshold']),
    onlyClusters=False,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"mutclust - ALL\"]/train_RAS", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"mutclust - ALL\"]/train_nonRAS", index=False)
test_RAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"mutclust - ALL\"]/test_RAS", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"mutclust - ALL\"]/test_nonRAS", index=False)

#### with variants - with only clusters

In [ ]:
# train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
#     sample_pos=samples_RAS,
#     sample_neg=samples_nonRAS,
#     sample_train=samples_train,
#     sample_test=samples_test,
#     mutations=mutclust_mut_merged,
#     includeVariants=True,
#     clusters=mutclust_clust,
#     clusters_threshold=float(data['mutclustSignificanceThreshold']),
#     onlyClusters=True,
#     seed=int(data['seed'])
# )

In [ ]:
# train_RAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"onlyMutclust - ALL\"]/train_RAS", index=False)
# train_nonRAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"onlyMutclust - ALL\"]/train_nonRAS", index=False)
# test_RAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"onlyMutclust - ALL\"]/test_RAS", index=False)
# test_nonRAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"onlyMutclust - ALL\"]/test_nonRAS", index=False)

### with Oncodriveclustl's clusters (deafult parameters)

In [ ]:
if 'oncodriveclustl_dp_mut_merged' not in locals():
    oncodriveclustl_mut_merged = mutations_merged.rename(columns={'Tumor_Sample_Barcode' : 'Sample', 'Hugo_Symbol' : 'name', 'Start_Position' : 'start', 'End_Position' : 'end', 'Variant_Classification': 'var'})
    oncodriveclustl_mut_merged['merge'] = oncodriveclustl_mut_merged['name']
    oncodriveclustl_mut_merged = oncodriveclustl_mut_merged[['Sample', 'name', 'merge', 'start', 'end', 'var']]

if 'oncodriveclustl_dp_clust' not in locals():
    oncodriveclustl_dp_clust = pd.read_csv(f"../{data['oncodriveclustl___all_noHypoHyper']}output_defaultParams/clusters_results.tsv", sep='\t', engine='python', comment='#')
    oncodriveclustl_dp_clust[['start', 'end']] = oncodriveclustl_dp_clust['COORDINATES'].str.split(',', expand=True)
    oncodriveclustl_dp_clust['start'] = oncodriveclustl_dp_clust['start'].astype(int)
    oncodriveclustl_dp_clust['end'] = oncodriveclustl_dp_clust['end'].astype(int)
    oncodriveclustl_dp_clust = oncodriveclustl_dp_clust.rename(columns={'SYMBOL' : 'merge'})
    oncodriveclustl_dp_clust = oncodriveclustl_dp_clust[['merge', 'start', 'end', 'P']]

#### without variants - with both clusters and non-clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=oncodriveclustl_mut_merged,
    includeVariants=False,
    clusters=oncodriveclustl_dp_clust,
    clusters_threshold=float(data['oncodriveclustlSignificanceThreshold']),
    onlyClusters=False,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}merged_mut_[\"oncodriveclustl_defaultParams - ALL\"]/train_RAS", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}merged_mut_[\"oncodriveclustl_defaultParams - ALL\"]/train_nonRAS", index=False)
test_RAS.to_csv(f"../{data['ml___input']}merged_mut_[\"oncodriveclustl_defaultParams - ALL\"]/test_RAS", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}merged_mut_[\"oncodriveclustl_defaultParams - ALL\"]/test_nonRAS", index=False)

#### without variants - with only clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=oncodriveclustl_mut_merged,
    includeVariants=False,
    clusters=oncodriveclustl_dp_clust,
    clusters_threshold=float(data['oncodriveclustlSignificanceThreshold']),
    onlyClusters=True,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}merged_mut_[\"onlyOncodriveclustl_defaultParams - ALL\"]/train_RAS", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}merged_mut_[\"onlyOncodriveclustl_defaultParams - ALL\"]/train_nonRAS", index=False)
test_RAS.to_csv(f"../{data['ml___input']}merged_mut_[\"onlyOncodriveclustl_defaultParams - ALL\"]/test_RAS", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}merged_mut_[\"onlyOncodriveclustl_defaultParams - ALL\"]/test_nonRAS", index=False)

### with Oncodriveclustl's clusters (colorectal parameters)

In [ ]:
if 'oncodriveclustl_mut_merged' not in locals():
    oncodriveclustl_mut_merged = mutations_merged.rename(columns={'Tumor_Sample_Barcode' : 'Sample', 'Hugo_Symbol' : 'name', 'Start_Position' : 'start', 'End_Position' : 'end', 'Variant_Classification': 'var'})
    oncodriveclustl_mut_merged['merge'] = oncodriveclustl_mut_merged['name']
    oncodriveclustl_mut_merged = oncodriveclustl_mut_merged[['Sample', 'name', 'merge', 'start', 'end', 'var']]

if 'oncodriveclustl_cp_clust' not in locals() or True:
    oncodriveclustl_cp_clust = pd.read_csv(f"../{data['oncodriveclustl___all_noHypoHyper']}output_coadreadParams/clusters_results.tsv", sep='\t', engine='python', comment='#')
    oncodriveclustl_cp_clust[['start', 'end']] = oncodriveclustl_cp_clust['COORDINATES'].str.split(',', expand=True)
    oncodriveclustl_cp_clust['start'] = oncodriveclustl_cp_clust['start'].astype(int)
    oncodriveclustl_cp_clust['end'] = oncodriveclustl_cp_clust['end'].astype(int)
    oncodriveclustl_cp_clust = oncodriveclustl_cp_clust.rename(columns={'SYMBOL' : 'merge'})
    oncodriveclustl_cp_clust = oncodriveclustl_cp_clust[['merge', 'start', 'end', 'P']]

#### without variants - with both clusters and non-clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=oncodriveclustl_mut_merged,
    includeVariants=False,
    clusters=oncodriveclustl_cp_clust,
    clusters_threshold=float(data['oncodriveclustlSignificanceThreshold']),
    onlyClusters=False,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}merged_mut_[\"oncodriveclustl_coadreadParams - ALL\"]/train_RAS", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}merged_mut_[\"oncodriveclustl_coadreadParams - ALL\"]/train_nonRAS", index=False)
test_RAS.to_csv(f"../{data['ml___input']}merged_mut_[\"oncodriveclustl_coadreadParams - ALL\"]/test_RAS", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}merged_mut_[\"oncodriveclustl_coadreadParams - ALL\"]/test_nonRAS", index=False)

#### without variants - with only clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=oncodriveclustl_mut_merged,
    includeVariants=False,
    clusters=oncodriveclustl_cp_clust,
    clusters_threshold=float(data['oncodriveclustlSignificanceThreshold']),
    onlyClusters=True,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}merged_mut_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/train_RAS", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}merged_mut_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/train_nonRAS", index=False)
test_RAS.to_csv(f"../{data['ml___input']}merged_mut_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/test_RAS", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}merged_mut_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/test_nonRAS", index=False)

#### with variants - with both clusters and non-clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=oncodriveclustl_mut_merged,
    includeVariants=True,
    clusters=oncodriveclustl_cp_clust,
    clusters_threshold=float(data['oncodriveclustlSignificanceThreshold']),
    onlyClusters=False,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"oncodriveclustl_coadreadParams - ALL\"]/train_RAS", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"oncodriveclustl_coadreadParams - ALL\"]/train_nonRAS", index=False)
test_RAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"oncodriveclustl_coadreadParams - ALL\"]/test_RAS", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"oncodriveclustl_coadreadParams - ALL\"]/test_nonRAS", index=False)

In [ ]:
train_RAS

#### with variants - with only clusters

In [ ]:
# train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
#     sample_pos=samples_RAS,
#     sample_neg=samples_nonRAS,
#     sample_train=samples_train,
#     sample_test=samples_test,
#     mutations=oncodriveclustl_mut_merged,
#     includeVariants=True,
#     clusters=oncodriveclustl_cp_clust,
#     clusters_threshold=float(data['oncodriveclustlSignificanceThreshold']),
#     onlyClusters=True,
#     seed=int(data['seed'])
# )

In [ ]:
# train_RAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/train_RAS", index=False)
# train_nonRAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/train_nonRAS", index=False)
# test_RAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/test_RAS", index=False)
# test_nonRAS.to_csv(f"../{data['ml___input']}merged_mutVar_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/test_nonRAS", index=False)

# TCGA dataset

In [ ]:
mutations_TCGA = pd.read_csv('../' + data['TCGA_noHypoHyper___mutations'], sep='\t', engine='python', comment='#')
mutations_TCGA = mutations_TCGA[~mutations_TCGA['Hugo_Symbol'].isin(['KRAS', 'HRAS', 'NRAS'])]

In [ ]:
transcriptomics_TCGA = pd.read_csv("../data/transcriptomics/TCGA_GMQL_GRCH38_FILTERED_unif_cpm.txt", sep='\t', comment='#', engine='python')
transcriptomics_TCGA = transcriptomics_TCGA.T.reset_index().rename(columns={'index' : 'Sample'})
transcriptomics_TCGA['Sample'] = transcriptomics_TCGA['Sample'].apply(lambda x: '-'.join(x.split('-')[:4])[:-1])
transcriptomics_TCGA = transcriptomics_TCGA.drop(columns=['KRAS', 'HRAS', 'NRAS'])
transcriptomics_TCGA = transcriptomics_TCGA.set_index('Sample')

## only mutational data

### with Mutclust's clusters

In [ ]:
if 'mutclust_mut_TCGA' not in locals():
    mutclust_mut_TCGA = mutations_TCGA.rename(columns={'Tumor_Sample_Barcode' : 'Sample', 'Hugo_Symbol' : 'name', 'Transcript_ID': 'merge', 'Protein_position' : 'start', 'Variant_Classification': 'var'})
    mutclust_mut_TCGA['end'] = mutclust_mut_TCGA['start']
    mutclust_mut_TCGA = mutclust_mut_TCGA[['Sample', 'name', 'merge', 'start', 'end', 'var']]

if 'mutclust_clust' not in locals():
    mutclust_clust = pd.read_csv('../' + data['mutclustsw___all_noHypoHyper___out'], sep='\t', engine='python', comment='#')
    mutclust_clust = mutclust_clust.rename(columns={'Gene' : 'merge', 'Start' : 'start', 'End' : 'end', 'Pvalue' : 'P'})
    mutclust_clust = mutclust_clust[['merge', 'start', 'end', 'P']]

#### without variants - with both clusters and non-clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=mutclust_mut_TCGA,
    includeVariants=False,
    clusters=mutclust_clust,
    clusters_threshold=float(data['mutclustSignificanceThreshold']),
    onlyClusters=False,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"mutclust - ALL\"]/train_RAS.csv", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"mutclust - ALL\"]/train_nonRAS.csv", index=False)
test_RAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"mutclust - ALL\"]/test_RAS.csv", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"mutclust - ALL\"]/test_nonRAS.csv", index=False)

#### without variants - with only clusters

In [ ]:
# train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
#     sample_pos=samples_RAS,
#     sample_neg=samples_nonRAS,
#     sample_train=samples_train,
#     sample_test=samples_test,
#     mutations=mutclust_mut_TCGA,
#     includeVariants=False,
#     clusters=mutclust_clust,
#     clusters_threshold=float(data['mutclustSignificanceThreshold']),
#     onlyClusters=True,
#     seed=int(data['seed'])
# )

In [ ]:
# train_RAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"onlyMutclust - ALL\"]/train_RAS", index=False)
# train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"onlyMutclust - ALL\"]/train_nonRAS", index=False)
# test_RAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"onlyMutclust - ALL\"]/test_RAS", index=False)
# test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"onlyMutclust - ALL\"]/test_nonRAS", index=False)

#### with variants - with both clusters and non-clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=mutclust_mut_TCGA,
    includeVariants=True,
    clusters=mutclust_clust,
    clusters_threshold=float(data['mutclustSignificanceThreshold']),
    onlyClusters=False,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"mutclust - ALL\"]/train_RAS.csv", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"mutclust - ALL\"]/train_nonRAS.csv", index=False)
test_RAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"mutclust - ALL\"]/test_RAS.csv", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"mutclust - ALL\"]/test_nonRAS.csv", index=False)

#### with variants - with only clusters

In [ ]:
# train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
#     sample_pos=samples_RAS,
#     sample_neg=samples_nonRAS,
#     sample_train=samples_train,
#     sample_test=samples_test,
#     mutations=mutclust_mut_TCGA,
#     includeVariants=True,
#     clusters=mutclust_clust,
#     clusters_threshold=float(data['mutclustSignificanceThreshold']),
#     onlyClusters=True,
#     seed=int(data['seed'])
# )

In [ ]:
# train_RAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"onlyMutclust - ALL\"]/train_RAS", index=False)
# train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"onlyMutclust - ALL\"]/train_nonRAS", index=False)
# test_RAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"onlyMutclust - ALL\"]/test_RAS", index=False)
# test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"onlyMutclust - ALL\"]/test_nonRAS", index=False)

### with Oncodriveclustl's clusters (colorectal parameters)

In [ ]:
if 'oncodriveclustl_mut_TCGA' not in locals():
    oncodriveclustl_mut_TCGA = mutations_TCGA.rename(columns={'Tumor_Sample_Barcode' : 'Sample', 'Hugo_Symbol' : 'name', 'Start_Position' : 'start', 'End_Position' : 'end', 'Variant_Classification': 'var'})
    oncodriveclustl_mut_TCGA['merge'] = oncodriveclustl_mut_TCGA['name']
    oncodriveclustl_mut_TCGA = oncodriveclustl_mut_TCGA[['Sample', 'name', 'merge', 'start', 'end', 'var']]

if 'oncodriveclustl_cp_clust' not in locals():
    oncodriveclustl_cp_clust = pd.read_csv(f"../{data['oncodriveclustl___all_noHypoHyper']}output_coadreadParams/clusters_results.tsv", sep='\t', engine='python', comment='#')
    oncodriveclustl_cp_clust[['start', 'end']] = oncodriveclustl_cp_clust['COORDINATES'].str.split(',', expand=True)
    oncodriveclustl_cp_clust['start'] = oncodriveclustl_cp_clust['start'].astype(int)
    oncodriveclustl_cp_clust['end'] = oncodriveclustl_cp_clust['end'].astype(int)
    oncodriveclustl_cp_clust = oncodriveclustl_cp_clust.rename(columns={'SYMBOL' : 'merge'})
    oncodriveclustl_cp_clust = oncodriveclustl_cp_clust[['merge', 'start', 'end', 'P']]

#### without variants - with both clusters and non-clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=oncodriveclustl_mut_TCGA,
    includeVariants=False,
    clusters=oncodriveclustl_cp_clust,
    clusters_threshold=float(data['oncodriveclustlSignificanceThreshold']),
    onlyClusters=False,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"oncodriveclustl_coadreadParams - ALL\"]/train_RAS.csv", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"oncodriveclustl_coadreadParams - ALL\"]/train_nonRAS.csv", index=False)
test_RAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"oncodriveclustl_coadreadParams - ALL\"]/test_RAS.csv", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"oncodriveclustl_coadreadParams - ALL\"]/test_nonRAS.csv", index=False)

#### without variants - with only clusters

In [ ]:
# train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
#     sample_pos=samples_RAS,
#     sample_neg=samples_nonRAS,
#     sample_train=samples_train,
#     sample_test=samples_test,
#     mutations=oncodriveclustl_mut_TCGA,
#     includeVariants=False,
#     clusters=oncodriveclustl_cp_clust,
#     clusters_threshold=float(data['oncodriveclustlSignificanceThreshold']),
#     onlyClusters=True,
#     seed=int(data['seed'])
# )

In [ ]:
# train_RAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/train_RAS", index=False)
# train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/train_nonRAS", index=False)
# test_RAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/test_RAS", index=False)
# test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mut_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/test_nonRAS", index=False)

#### with variants - with both clusters and non-clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=oncodriveclustl_mut_TCGA,
    includeVariants=True,
    clusters=oncodriveclustl_cp_clust,
    clusters_threshold=float(data['oncodriveclustlSignificanceThreshold']),
    onlyClusters=False,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"oncodriveclustl_coadreadParams - ALL\"]/train_RAS.csv", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"oncodriveclustl_coadreadParams - ALL\"]/train_nonRAS.csv", index=False)
test_RAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"oncodriveclustl_coadreadParams - ALL\"]/test_RAS.csv", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"oncodriveclustl_coadreadParams - ALL\"]/test_nonRAS.csv", index=False)

#### with variants - with only clusters

In [ ]:
# train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
#     sample_pos=samples_RAS,
#     sample_neg=samples_nonRAS,
#     sample_train=samples_train,
#     sample_test=samples_test,
#     mutations=oncodriveclustl_mut_TCGA,
#     includeVariants=True,
#     clusters=oncodriveclustl_cp_clust,
#     clusters_threshold=float(data['oncodriveclustlSignificanceThreshold']),
#     onlyClusters=True,
#     seed=int(data['seed'])
# )

In [ ]:
# train_RAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/train_RAS", index=False)
# train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/train_nonRAS", index=False)
# test_RAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/test_RAS", index=False)
# test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_mutVar_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/test_nonRAS", index=False)

## only transcriptomic data

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    transcriptomics=transcriptomics_TCGA,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}TCGA_tran/train_RAS", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran/train_nonRAS", index=False)
test_RAS.to_csv(f"../{data['ml___input']}TCGA_tran/test_RAS", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran/test_nonRAS", index=False)

## Transcriptomic and mutational data

### with Mutclust's clusters

In [ ]:
if 'mutclust_mut_TCGA' not in locals():
    mutclust_mut_TCGA = mutations_TCGA.rename(columns={'Tumor_Sample_Barcode' : 'Sample', 'Hugo_Symbol' : 'name', 'Transcript_ID': 'merge', 'Protein_position' : 'start', 'Variant_Classification': 'var'})
    mutclust_mut_TCGA['end'] = mutclust_mut_TCGA['start']
    mutclust_mut_TCGA = mutclust_mut_TCGA[['Sample', 'name', 'merge', 'start', 'end', 'var']]

if 'mutclust_clust' not in locals():
    mutclust_clust = pd.read_csv('../' + data['mutclustsw___all_noHypoHyper___out'], sep='\t', engine='python', comment='#')
    mutclust_clust = mutclust_clust.rename(columns={'Gene' : 'merge', 'Start' : 'start', 'End' : 'end', 'Pvalue' : 'P'})
    mutclust_clust = mutclust_clust[['merge', 'start', 'end', 'P']]

#### without variants - with both clusters and non-clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=mutclust_mut_TCGA,
    includeVariants=False,
    clusters=mutclust_clust,
    clusters_threshold=float(data['mutclustSignificanceThreshold']),
    onlyClusters=False,
    transcriptomics=transcriptomics_TCGA,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"mutclust - ALL\"]/train_RAS.csv", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"mutclust - ALL\"]/train_nonRAS.csv", index=False)
test_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"mutclust - ALL\"]/test_RAS.csv", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"mutclust - ALL\"]/test_nonRAS.csv", index=False)

#### without variants - with only clusters

In [ ]:
# train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
#     sample_pos=samples_RAS,
#     sample_neg=samples_nonRAS,
#     sample_train=samples_train,
#     sample_test=samples_test,
#     mutations=mutclust_mut_TCGA,
#     includeVariants=False,
#     clusters=mutclust_clust,
#     clusters_threshold=float(data['mutclustSignificanceThreshold']),
#     onlyClusters=True,
#     transcriptomics=transcriptomics_TCGA,
#     seed=int(data['seed'])
# )

In [ ]:
# train_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"onlyMutclust - ALL\"]/train_RAS", index=False)
# train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"onlyMutclust - ALL\"]/train_nonRAS", index=False)
# test_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"onlyMutclust - ALL\"]/test_RAS", index=False)
# test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"onlyMutclust - ALL\"]/test_nonRAS", index=False)

#### with variants - with both clusters and non-clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=mutclust_mut_TCGA,
    includeVariants=True,
    clusters=mutclust_clust,
    clusters_threshold=float(data['mutclustSignificanceThreshold']),
    onlyClusters=False,
    transcriptomics=transcriptomics_TCGA,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"mutclust - ALL\"]/train_RAS.csv", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"mutclust - ALL\"]/train_nonRAS.csv", index=False)
test_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"mutclust - ALL\"]/test_RAS.csv", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"mutclust - ALL\"]/test_nonRAS.csv", index=False)

#### with variants - with only clusters

In [ ]:
# train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
#     sample_pos=samples_RAS,
#     sample_neg=samples_nonRAS,
#     sample_train=samples_train,
#     sample_test=samples_test,
#     mutations=mutclust_mut_TCGA,
#     includeVariants=True,
#     clusters=mutclust_clust,
#     clusters_threshold=float(data['mutclustSignificanceThreshold']),
#     onlyClusters=True,
#     transcriptomics=transcriptomics_TCGA,
#     seed=int(data['seed'])
# )

In [ ]:
# train_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"onlyMutclust - ALL\"]/train_RAS", index=False)
# train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"onlyMutclust - ALL\"]/train_nonRAS", index=False)
# test_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"onlyMutclust - ALL\"]/test_RAS", index=False)
# test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"onlyMutclust - ALL\"]/test_nonRAS", index=False)

### with Oncodriveclustl's clusters (colorectal parameters)

In [ ]:
if 'oncodriveclustl_mut_TCGA' not in locals():
    oncodriveclustl_mut_TCGA = mutations_TCGA.rename(columns={'Tumor_Sample_Barcode' : 'Sample', 'Hugo_Symbol' : 'name', 'Start_Position' : 'start', 'End_Position' : 'end', 'Variant_Classification': 'var'})
    oncodriveclustl_mut_TCGA['merge'] = oncodriveclustl_mut_TCGA['name']
    oncodriveclustl_mut_TCGA = oncodriveclustl_mut_TCGA[['Sample', 'name', 'merge', 'start', 'end', 'var']]

if 'oncodriveclustl_cp_clust' not in locals():
    oncodriveclustl_cp_clust = pd.read_csv(f"../{data['oncodriveclustl___all_noHypoHyper']}output_coadreadParams/clusters_results.tsv", sep='\t', engine='python', comment='#')
    oncodriveclustl_cp_clust[['start', 'end']] = oncodriveclustl_cp_clust['COORDINATES'].str.split(',', expand=True)
    oncodriveclustl_cp_clust['start'] = oncodriveclustl_cp_clust['start'].astype(int)
    oncodriveclustl_cp_clust['end'] = oncodriveclustl_cp_clust['end'].astype(int)
    oncodriveclustl_cp_clust = oncodriveclustl_cp_clust.rename(columns={'SYMBOL' : 'merge'})
    oncodriveclustl_cp_clust = oncodriveclustl_cp_clust[['merge', 'start', 'end', 'P']]

#### without variants - with both clusters and non-clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=oncodriveclustl_mut_TCGA,
    includeVariants=False,
    clusters=oncodriveclustl_cp_clust,
    clusters_threshold=float(data['oncodriveclustlSignificanceThreshold']),
    onlyClusters=False,
    transcriptomics=transcriptomics_TCGA,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"oncodriveclustl_coadreadParams - ALL\"]/train_RAS.csv", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"oncodriveclustl_coadreadParams - ALL\"]/train_nonRAS.csv", index=False)
test_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"oncodriveclustl_coadreadParams - ALL\"]/test_RAS.csv", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"oncodriveclustl_coadreadParams - ALL\"]/test_nonRAS.csv", index=False)

#### without variants - with only clusters

In [ ]:
# train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
#     sample_pos=samples_RAS,
#     sample_neg=samples_nonRAS,
#     sample_train=samples_train,
#     sample_test=samples_test,
#     mutations=oncodriveclustl_mut_TCGA,
#     includeVariants=False,
#     clusters=oncodriveclustl_cp_clust,
#     clusters_threshold=float(data['oncodriveclustlSignificanceThreshold']),
#     onlyClusters=True,
#     transcriptomics=transcriptomics_TCGA,
#     seed=int(data['seed'])
# )

In [ ]:
# train_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/train_RAS", index=False)
# train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/train_nonRAS", index=False)
# test_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/test_RAS", index=False)
# test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mut_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/test_nonRAS", index=False)

#### with variants - with both clusters and non-clusters

In [ ]:
train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
    sample_pos=samples_RAS,
    sample_neg=samples_nonRAS,
    sample_train=samples_train,
    sample_test=samples_test,
    mutations=oncodriveclustl_mut_TCGA,
    includeVariants=True,
    clusters=oncodriveclustl_cp_clust,
    clusters_threshold=float(data['oncodriveclustlSignificanceThreshold']),
    onlyClusters=False,
    transcriptomics=transcriptomics_TCGA,
    seed=int(data['seed'])
)

In [ ]:
train_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"oncodriveclustl_coadreadParams - ALL\"]/train_RAS.csv", index=False)
train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"oncodriveclustl_coadreadParams - ALL\"]/train_nonRAS.csv", index=False)
test_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"oncodriveclustl_coadreadParams - ALL\"]/test_RAS.csv", index=False)
test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"oncodriveclustl_coadreadParams - ALL\"]/test_nonRAS.csv", index=False)

#### with variants - with only clusters

In [ ]:
# train_RAS, train_nonRAS, test_RAS, test_nonRAS = prepareInput(
#     sample_pos=samples_RAS,
#     sample_neg=samples_nonRAS,
#     sample_train=samples_train,
#     sample_test=samples_test,
#     mutations=oncodriveclustl_mut_TCGA,
#     includeVariants=True,
#     clusters=oncodriveclustl_cp_clust,
#     clusters_threshold=float(data['oncodriveclustlSignificanceThreshold']),
#     onlyClusters=True,
#     transcriptomics=transcriptomics_TCGA,
#     seed=int(data['seed'])
# )

In [ ]:
# train_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/train_RAS", index=False)
# train_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/train_nonRAS", index=False)
# test_RAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/test_RAS", index=False)
# test_nonRAS.to_csv(f"../{data['ml___input']}TCGA_tran&mutVar_[\"onlyOncodriveclustl_coadreadParams - ALL\"]/test_nonRAS", index=False)